# Banco Central / crédito rural
Acesso público à **MDCR / SICOR v2 (Olinda)** para custeio agrícola de soja, milho e cana-de-açúcar.

| Recurso | Uso no projeto |
|---|---|
| `CusteioMunicipioProduto` | Valor por município e cultura: perguntas 2–7 e extra B, após integrar a PAM. |
| `CusteioRegiaoUFProduto` | Valor e quantidade de contratos por UF/região e cultura: pergunta 1 e visão nacional da 2. |

**Limitação:** o recurso municipal por produto não possui `QtdCusteio`.


## SETUP - Essa seção do código é apenas o backbone para podermos construir as visualizações mais facilmente.

In [ ]:
import hashlib
import json
import unicodedata # normalizar texto que vai/volta de/para api
import xml.etree.ElementTree as ET # nao me pergunte o que isso faz
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote, urlencode

import pandas as pd
import requests
from IPython.display import display # biblioteca legal para visualizações no notebook
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE = "https://olinda.bcb.gov.br/olinda/servico/SICOR/versao/v2/odata/"
RECURSO_MUNICIPAL = "CusteioMunicipioProduto"
RECURSO_UF = "CusteioRegiaoUFProduto"
RECURSOS = (RECURSO_MUNICIPAL, RECURSO_UF)
CACHE = Path("dados/bcb/cache")
TIMEOUT = (10, 90)  # conexão, leitura (segundos)
LIMITE_LOTE = 100_000
ANO_EXEMPLO, MES_EXEMPLO = 2025, 1  # teste; não define a década
ATIVIDADE_AGRICOLA = "1"

sessao = requests.Session()
sessao.headers.update({"User-Agent": "AED-Credito-Rural/1.0"})
retry = Retry(
    total=3, backoff_factor=1,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset({"GET"}), respect_retry_after_header=True,
)
sessao.mount("https://olinda.bcb.gov.br/", HTTPAdapter(max_retries=retry))


#### SETUP - Cliente HTTP e cache auditável


In [2]:
def literal_odata(valor):
    return "'" + str(valor).replace("'", "''") + "'"


def obter_resposta(url):
    try:
        resposta = sessao.get(url, timeout=TIMEOUT)
        resposta.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(f"Falha ao consultar BCB: {url}\n{exc}") from exc
    return resposta


def consultar(recurso, filtro=None, *, limite=5, completo=False, atualizar=False):
    if recurso not in RECURSOS:
        raise ValueError(f"Recurso não previsto: {recurso}")
    if not isinstance(limite, int) or not 1 <= limite <= LIMITE_LOTE:
        raise ValueError(f"limite deve estar entre 1 e {LIMITE_LOTE}")
    params = {"$format": "json", "$top": limite}
    if filtro:
        params["$filter"] = filtro
    url = BASE + recurso + "?" + urlencode(params, quote_via=quote)
    arquivo = CACHE / (hashlib.sha256(url.encode()).hexdigest() + ".json")
    if arquivo.exists() and not atualizar:
        registro = json.loads(arquivo.read_text(encoding="utf-8"))
        if registro["url"] != url:
            raise ValueError("Cache incompatível com a consulta.")
    else:
        resposta = obter_resposta(url)
        try:
            dados = resposta.json()
        except ValueError as exc:
            raise ValueError("O BCB não retornou JSON válido.") from exc
        if not isinstance(dados, dict) or not isinstance(dados.get("value"), list):
            raise ValueError("Resposta OData sem a lista 'value'.")
        registro = {
            "url": url, "obtido_em_utc": datetime.now(timezone.utc).isoformat(),
            "dados": dados,
        }
        CACHE.mkdir(parents=True, exist_ok=True)
        temporario = arquivo.with_suffix(".tmp")
        temporario.write_text(json.dumps(registro, ensure_ascii=False), encoding="utf-8")
        temporario.replace(arquivo)
    dados = registro["dados"]
    if not isinstance(dados.get("value"), list):
        raise ValueError("Cache inválido: lista 'value' ausente.")
    if completo and (len(dados["value"]) >= limite or dados.get("@odata.nextLink")):
        raise RuntimeError("Possível truncamento. Reduza o recorte; não use $skip.")
    df = pd.DataFrame(dados["value"])
    df.attrs.update({
        "url": url, "obtido_em_utc": registro["obtido_em_utc"],
        "consulta_completa": completo, "arquivo_cache": str(arquivo),
    })
    return df


#### SETUP - Conferir o esquema oficial
Identificamos o tipo associado a cada recurso sem pressupor seus nomes internos. A chave municipal para integração é `codIbge`, preservada como texto. `codCadMu` e `cdEstado` são códigos do BCB, não códigos IBGE.


In [3]:
def ler_esquema():
    xml = obter_resposta(BASE + "$metadata").content
    raiz = ET.fromstring(xml)
    ns = {"edm": "http://docs.oasis-open.org/odata/ns/edm"}
    tipos = {item.attrib["Name"]: item for item in raiz.findall(".//edm:EntityType", ns)}
    linhas = []
    for entidade in raiz.findall(".//edm:EntitySet", ns):
        recurso = entidade.attrib["Name"]
        if recurso in RECURSOS:
            tipo = tipos[entidade.attrib["EntityType"].rsplit(".", 1)[-1]]
            for campo in tipo.findall("edm:Property", ns):
                linhas.append({"recurso": recurso, "campo": campo.attrib["Name"],
                               "tipo": campo.attrib["Type"]})
    esquema = pd.DataFrame(linhas)
    if esquema.empty or set(esquema["recurso"]) != set(RECURSOS):
        raise ValueError("Recursos esperados não encontrados no esquema.")
    CACHE.mkdir(parents=True, exist_ok=True)
    (CACHE / "metadata.xml").write_bytes(xml)
    return esquema


esquema = ler_esquema()
display(esquema)
campos_municipais = set(esquema.loc[esquema["recurso"].eq(RECURSO_MUNICIPAL), "campo"])
campos_uf = set(esquema.loc[esquema["recurso"].eq(RECURSO_UF), "campo"])
assert {"codIbge", "nomeProduto", "VlCusteio", "AnoEmissao"} <= campos_municipais
assert {"nomeUF", "nomeRegiao", "QtdCusteio", "VlCusteio"} <= campos_uf
print("Quantidade de contratos no recurso municipal:", "QtdCusteio" in campos_municipais)


,recurso,campo,tipo
0,CusteioMunicipioProduto,Municipio,Edm.String
1,CusteioMunicipioProduto,nomeProduto,Edm.String
2,CusteioMunicipioProduto,MesEmissao,Edm.String
3,CusteioMunicipioProduto,AnoEmissao,Edm.String
4,CusteioMunicipioProduto,cdPrograma,Edm.String
5,CusteioMunicipioProduto,cdSubPrograma,Edm.String
6,CusteioMunicipioProduto,cdFonteRecurso,Edm.String
7,CusteioMunicipioProduto,cdTipoSeguro,Edm.String
8,CusteioMunicipioProduto,cdEstado,Edm.String
9,CusteioMunicipioProduto,VlCusteio,Edm.Decimal


Quantidade de contratos no recurso municipal: False


#### SETUP - Identificar culturas em pequenas amostras

Janeiro/2025 é apenas um teste.


In [ ]:
def normalizar_nome(valor):
    texto = unicodedata.normalize("NFKD", str(valor).strip().strip('"').upper())
    return " ".join("".join(c for c in texto if not unicodedata.combining(c))
                    .replace("-", " ").split())


CULTURAS = {"SOJA": "soja", "MILHO": "milho", "CANA DE ACUCAR": "cana_de_acucar"}


def filtro_periodo(ano, mes):
    if not isinstance(ano, int) or not 1900 <= ano <= 2100:
        raise ValueError("Ano inválido.")
    if not isinstance(mes, int) or not 1 <= mes <= 12:
        raise ValueError("Mês inválido.")
    return (f"AnoEmissao eq {literal_odata(ano)} and "
            f"MesEmissao eq {literal_odata(f'{mes:02d}')} and "
            f"Atividade eq {literal_odata(ATIVIDADE_AGRICOLA)}")


def descobrir_produtos(ano, mes):
    amostras = []
    for termo in ("SOJA", "MILHO", "CANA"):
        filtro = filtro_periodo(ano, mes) + f" and contains(nomeProduto,{literal_odata(termo)})"
        amostra = consultar(RECURSO_MUNICIPAL, filtro, limite=10)
        if not amostra.empty:
            amostras.append(amostra[["nomeProduto", "cdProduto", "Atividade"]])
    if not amostras:
        raise ValueError("Amostras vazias. Confira o período ou tente novamente.")
    catalogo = pd.concat(amostras, ignore_index=True).drop_duplicates()
    catalogo["cultura"] = catalogo["nomeProduto"].map(normalizar_nome).map(CULTURAS)
    return catalogo.reset_index(drop=True) # drop=True -> para tirar a coluna de indice anterior


catalogo_amostra = descobrir_produtos(ANO_EXEMPLO, MES_EXEMPLO)
display(catalogo_amostra)
produtos = (catalogo_amostra.dropna(subset=["cultura"])[["nomeProduto", "cultura"]]
            .drop_duplicates().reset_index(drop=True))
if set(produtos["cultura"]) != set(CULTURAS.values()):
    raise ValueError("A amostra não identificou as três culturas. Revise as categorias.")


,nomeProduto,cdProduto,Atividade,cultura
0,"""SOJA""",6720,1,soja
1,"""MILHO""",5080,1,milho
2,"""CANA-DE-AÇUCAR""",1840,1,cana_de_acucar


#### SETUP - Coleta por lote e preparação

In [5]:
def preparar_lote(bruto, recurso):
    if recurso not in RECURSOS:
        raise ValueError("Recurso fora do escopo.")
    if bruto.empty:
        return pd.DataFrame()
    obrigatorios = {"nomeProduto", "AnoEmissao", "MesEmissao", "Atividade", "VlCusteio"}
    obrigatorios |= ({"codIbge"} if recurso == RECURSO_MUNICIPAL
                     else {"nomeUF", "nomeRegiao", "QtdCusteio"})
    faltantes = obrigatorios - set(bruto.columns)
    if faltantes:
        raise ValueError(f"Campos ausentes: {sorted(faltantes)}")
    df = bruto.copy()
    df["cultura"] = df["nomeProduto"].map(normalizar_nome).map(CULTURAS)
    if df["cultura"].isna().any() or not df["Atividade"].eq("1").all():
        raise ValueError("Produto não mapeado ou atividade não agrícola.")
    for origem, destino in (("AnoEmissao", "ano"), ("MesEmissao", "mes")):
        df[destino] = pd.to_numeric(df[origem], errors="raise").astype("Int64")
        if df[destino].isna().any():
            raise ValueError(f"{origem} ausente.")
    if not df["mes"].between(1, 12).all():
        raise ValueError("Mês fora do intervalo 1–12.")
    df["credito_nominal_brl"] = pd.to_numeric(df["VlCusteio"], errors="raise").astype("Float64")
    if df["credito_nominal_brl"].dropna().lt(0).any():
        raise ValueError("Valor negativo: investigar antes de agregar.")
    if recurso == RECURSO_MUNICIPAL:
        df["codigo_municipio"] = df["codIbge"].astype("string").str.strip()
        if not df["codigo_municipio"].str.fullmatch(r"\d{7}", na=False).all():
            raise ValueError("Código IBGE ausente ou inválido; não preencher com zeros.")
        df["codigo_uf_ibge"] = df["codigo_municipio"].str[:2]
        df["quantidade_contratos"] = pd.Series(pd.NA, index=df.index, dtype="Int64")
    else:
        df["uf"] = df["nomeUF"].astype("string").str.strip()
        df["regiao"] = df["nomeRegiao"].astype("string").str.strip()
        if df[["uf", "regiao"]].isna().any().any() or df["uf"].eq("").any() or df["regiao"].eq("").any():
            raise ValueError("UF ou região ausente.")
        df["quantidade_contratos"] = pd.to_numeric(df["QtdCusteio"], errors="raise").astype("Int64")
        if df["quantidade_contratos"].dropna().lt(0).any():
            raise ValueError("Quantidade negativa.")
    return df


def coletar_lote(recurso, ano, mes, nome_produto, *, atualizar=False):
    if recurso not in RECURSOS or normalizar_nome(nome_produto) not in CULTURAS:
        raise ValueError("Recurso ou produto fora do escopo.")
    filtro = filtro_periodo(ano, mes) + f" and nomeProduto eq {literal_odata(nome_produto)}"
    bruto = consultar(recurso, filtro, limite=LIMITE_LOTE, completo=True, atualizar=atualizar)
    tratado = preparar_lote(bruto, recurso)
    if not tratado.empty:
        if not (tratado["ano"].eq(ano) & tratado["mes"].eq(mes)
                & tratado["nomeProduto"].eq(nome_produto)).all():
            raise ValueError("Resposta não respeita o filtro solicitado.")
    auditoria = {"recurso": recurso, "ano": ano, "mes": mes, "nome_produto": nome_produto,
                 "linhas": len(bruto), **bruto.attrs}
    return tratado, auditoria


def coletar_periodo(recurso, anos, produtos, *, atualizar=False):
    anos = list(anos)
    if not anos or len(anos) != len(set(anos)):
        raise ValueError("Informe anos distintos.")
    nomes = produtos["nomeProduto"].drop_duplicates().tolist()
    if not nomes:
        raise ValueError("Informe produtos validados.")
    lotes, auditoria = [], []
    for ano in sorted(anos):
        for mes in range(1, 13):
            for nome in nomes:
                lote, registro = coletar_lote(recurso, ano, mes, nome, atualizar=atualizar)
                auditoria.append(registro)  # inclui respostas vazias
                if not lote.empty:
                    lotes.append(lote)
        print(f"{recurso}: {ano} consultado.")
    dados = pd.concat(lotes, ignore_index=True) if lotes else pd.DataFrame()
    return dados, pd.DataFrame(auditoria)


def agregar_anual(dados, recurso):
    if recurso not in RECURSOS:
        raise ValueError("Recurso fora do escopo.")
    if dados.empty:
        raise ValueError("Sem dados para agregar.")
    chaves = (["codigo_municipio", "codigo_uf_ibge"] if recurso == RECURSO_MUNICIPAL
              else ["uf", "regiao"]) + ["cultura", "ano"]
    def somar_completos(serie):
        # Uma parcela ausente torna o total desconhecido, evitando subestimação.
        return serie.sum(min_count=len(serie))
    return (dados.groupby(chaves, as_index=False, dropna=False)[
                ["credito_nominal_brl", "quantidade_contratos"]]
            .agg(somar_completos).sort_values(chaves).reset_index(drop=True))


## VISUALIZAÇÕES
### Nesta seção serão construídas as visualizações que responderam as perguntas feitas em nosso relatório

import altair as alt
import numpy as np
